# Zonas Rurales España — Ley 45/2007
### Cruce de fuentes INE: Padrón Municipal 2025 + Grupos de edad 2022

**Objetivo:** Construir una tabla única con todos los municipios de España que permita:
- Clasificar qué municipios cumplen los criterios de zona rural (Ley 45/2007, art. 3)
- Conocer la población en edad laboral (16–64 años) y dependiente (65+) por municipio

**Criterio legal:**
> *"Se entiende por zona rural el espacio geográfico formado por la agregación de municipios o entidades locales menores definido por las administraciones competentes que posean una **población inferior a 30.000 habitantes** y una **densidad inferior a los 100 habitantes por km²**"*  
> — Ley 45/2007, de 13 de diciembre, art. 3

---

### Fuentes de datos utilizadas

| # | Fichero | Dato | Referencia | Publicado | Descarga |
|---|---------|------|-----------|-----------|---------|
| 1 | `pobmun25.xlsx` | Población total por municipio | 1 enero 2025 | 11/12/2025 | [ine.es/pob_xls/pobmun.zip](https://www.ine.es/pob_xls/pobmun.zip) |
| 2 | `33861.xlsx` | Grupos de edad: 0–15, 16–64, 65+ | 1 enero 2022 | 2023 | [ine.es/jaxiT3/Tabla.htm?t=33861](https://www.ine.es/jaxiT3/Tabla.htm?t=33861) |
| 3 | `municipios_ine.json` | Base completa preprocesada | Cruce 1+2 | Este proyecto | Generado por `scripts/build_json.py` |

> **Nota sobre superficie:** La superficie km² por municipio se obtiene de la **tabla 2879** del INE
> (Densidad de población por municipio) a través de la API JSON en tiempo real.  
> En este notebook se trabaja con los datos sin superficie para operar offline;  
> la celda [Opcional] al final muestra cómo añadirla desde la API.

---

### Requisitos
```bash
pip install pandas openpyxl requests
```

## 0. Importaciones y configuración

In [ ]:
import re
import json
import warnings
from pathlib import Path

import pandas as pd
import openpyxl

warnings.filterwarnings('ignore', category=UserWarning)  # openpyxl style warnings

# ── Rutas ──────────────────────────────────────────────────────────────────────
# Ajusta DATA_DIR si tus ficheros están en otra carpeta
DATA_DIR = Path('data')   # pobmun25.xlsx y 33861.xlsx
JSON_PATH = Path('public/municipios_ine.json')  # generado por build_json.py

print("pandas:", pd.__version__)
print("Ficheros en data/:")
for f in sorted(DATA_DIR.glob('*')):
    print(f"  {f.name}  ({f.stat().st_size/1024:.0f} KB)")

## 1. Cargar `pobmun25.xlsx` — Población oficial por municipio (1 enero 2025)

**Estructura del fichero:**
- Fila 1: título
- Fila 2: cabecera → `CPRO | PROVINCIA | CMUN | NOMBRE | POB25 | HOMBRES | MUJERES`
- Filas 3+: un municipio por fila

**Código INE:** 5 dígitos = 2 (provincia) + 3 (municipio).  
Ejemplo: `28079` = provincia Madrid (28) + municipio Madrid (079)

In [ ]:
# ── Encontrar el fichero pobmun más reciente ──────────────────────────────────
import zipfile

def cargar_pobmun(data_dir: Path) -> pd.DataFrame:
    """
    Carga el fichero pobmun más reciente de data_dir.
    Acepta: pobmun.zip (extrae el xlsx más nuevo dentro) o pobmun*.xlsx directamente.
    """
    # Buscar ZIP
    zips = sorted(data_dir.glob('pobmun*.zip'))
    if zips:
        z = zipfile.ZipFile(zips[-1])
        xlsx_inside = sorted([f for f in z.namelist() if f.endswith('.xlsx')])
        if xlsx_inside:
            newest = xlsx_inside[-1]
            tmp = data_dir / Path(newest).name
            with open(tmp, 'wb') as f:
                f.write(z.read(newest))
            print(f"ZIP: extraído {newest}")
            return _leer_pobmun_xlsx(tmp)

    # Buscar XLSX directamente
    xlsx = sorted(data_dir.glob('pobmun*.xlsx'))
    if xlsx:
        return _leer_pobmun_xlsx(xlsx[-1])

    raise FileNotFoundError(
        "No se encontró pobmun*.zip ni pobmun*.xlsx en data/\n"
        "Descarga desde: https://www.ine.es/pob_xls/pobmun.zip"
    )


def _leer_pobmun_xlsx(path: Path) -> pd.DataFrame:
    print(f"Leyendo: {path.name}")
    wb = openpyxl.load_workbook(path, read_only=True, data_only=True)
    ws = wb.active
    rows = list(ws.iter_rows(values_only=True))
    wb.close()

    header = rows[1]                              # fila 2 = cabecera
    col_pob = next(i for i, h in enumerate(header) if h and str(h).startswith('POB'))
    anio = '20' + str(header[col_pob])[3:]        # "POB25" → "2025"

    data = []
    for row in rows[2:]:
        if row[0] is None or row[2] is None or row[col_pob] is None:
            continue
        cpro = str(row[0]).strip().zfill(2)
        cmun = str(row[2]).strip().zfill(3)
        data.append({
            'cod_ine':   cpro + cmun,
            'cpro':      cpro,
            'cmun':      cmun,
            'nombre':    str(row[3]).strip() if row[3] else '',
            'provincia': str(row[1]).strip() if row[1] else '',
            'pob_total': int(row[col_pob]),
            'hombres':   int(row[5]) if row[5] is not None else None,
            'mujeres':   int(row[6]) if row[6] is not None else None,
            'anio_pob':  anio,
        })

    df = pd.DataFrame(data)
    print(f"  → {len(df):,} municipios  |  año de referencia: {anio}")
    return df


df_pob = cargar_pobmun(DATA_DIR)
df_pob.head(3)

In [ ]:
# Validación rápida
print(f"Municipios totales:     {len(df_pob):,}")
print(f"Provincias:             {df_pob['cpro'].nunique()}")
print(f"Municipios sin nombre:  {df_pob['nombre'].eq('').sum()}")
print(f"Municipios pob=0:       {(df_pob['pob_total'] == 0).sum()}")
print(f"Población total España: {df_pob['pob_total'].sum():,}")
print()
print("Distribución por tamaño de municipio:")
bins = [0, 100, 500, 1000, 5000, 10000, 30000, 100000, float('inf')]
labels = ['<100', '100-500', '500-1K', '1K-5K', '5K-10K', '10K-30K', '30K-100K', '>100K']
df_pob['tramo'] = pd.cut(df_pob['pob_total'], bins=bins, labels=labels, right=True)
print(df_pob['tramo'].value_counts().sort_index().to_string())

## 2. Añadir CCAA a partir del código de provincia

In [ ]:
# Tabla oficial de correspondencia provincia → CCAA
# Fuente: INE - Relación de municipios y sus códigos por provincias
PROV_CCAA = {
    '01': 'País Vasco',             '02': 'Castilla-La Mancha',
    '03': 'Comunitat Valenciana',   '04': 'Andalucía',
    '05': 'Castilla y León',        '06': 'Extremadura',
    '07': 'Illes Balears',          '08': 'Cataluña',
    '09': 'Castilla y León',        '10': 'Extremadura',
    '11': 'Andalucía',              '12': 'Comunitat Valenciana',
    '13': 'Castilla-La Mancha',     '14': 'Andalucía',
    '15': 'Galicia',                '16': 'Castilla-La Mancha',
    '17': 'Cataluña',               '18': 'Andalucía',
    '19': 'Castilla-La Mancha',     '20': 'País Vasco',
    '21': 'Andalucía',              '22': 'Aragón',
    '23': 'Andalucía',              '24': 'Castilla y León',
    '25': 'Cataluña',               '26': 'La Rioja',
    '27': 'Galicia',                '28': 'Comunidad de Madrid',
    '29': 'Andalucía',              '30': 'Región de Murcia',
    '31': 'Navarra',                '32': 'Galicia',
    '33': 'Asturias',               '34': 'Castilla y León',
    '35': 'Canarias',               '36': 'Galicia',
    '37': 'Castilla y León',        '38': 'Canarias',
    '39': 'Cantabria',              '40': 'Castilla y León',
    '41': 'Andalucía',              '42': 'Castilla y León',
    '43': 'Cataluña',               '44': 'Aragón',
    '45': 'Castilla-La Mancha',     '46': 'Comunitat Valenciana',
    '47': 'Castilla y León',        '48': 'País Vasco',
    '49': 'Castilla y León',        '50': 'Aragón',
    '51': 'Ceuta',                  '52': 'Melilla',
}

df_pob['ccaa'] = df_pob['cpro'].map(PROV_CCAA)

print("Municipios por CCAA:")
print(df_pob.groupby('ccaa')['cod_ine'].count().sort_values(ascending=False).to_string())

## 3. Cargar `33861.xlsx` — Grupos de edad por municipio

**Estructura del fichero:**
- El fichero está organizado como serie temporal: cada municipio tiene un bloque
  con sus datos para cada año (2003–2022)
- **Solo contiene una provincia**: la que seleccionaste al descargar desde INEbase  
  (en este caso: Navarra, provincia 31)
- Para tener datos nacionales necesitas descargar el fichero **por cada provincia** 
  y concatenarlos, o usar la descarga nacional completa disponible en INEbase > 
  Padrón Continuo > Tabla 33861 > seleccionar "Nacional" > Descargar como Excel

**Columnas devueltas (Total = españoles + extranjeros, ambos sexos):**

| Col | Contenido |
|-----|-----------|
| B | Total todas las edades |
| C | Menos de 16 años |
| D | De 16 a 64 años ← edad laboral |
| E | 65 y más años |
| F–I | Mismos grupos, solo españoles |
| J–M | Mismos grupos, solo extranjeros |

In [ ]:
def cargar_33861(path: Path) -> pd.DataFrame:
    """
    Parsea el fichero 33861.xlsx del INE.

    Estructura del fichero:
      - Cabecera en filas 7-8 (doble nivel: Total/Española/Extranjera + grupos de edad)
      - Municipios identificados por '    {5 dígitos} {nombre}'  (4+ espacios al inicio)
      - Bajo cada municipio: una fila por año con los datos numéricos
      - Solo se toma el año más reciente disponible (primera fila de datos = más reciente)

    Devuelve un DataFrame con una fila por municipio y el dato más reciente.
    """
    print(f"Leyendo: {path.name}")
    wb = openpyxl.load_workbook(path, read_only=True, data_only=True)
    ws = wb.active
    rows = list(ws.iter_rows(values_only=True))
    wb.close()

    records = []
    current_cod = None
    current_nombre = None
    data_taken = False

    for row in rows:
        cell = str(row[0] or '').rstrip()

        # Detectar fila de municipio: ≥4 espacios + código 5 dígitos + nombre
        m = re.match(r'^\s{4,}(\d{5})\s+(.+)$', cell)
        if m:
            current_cod    = m.group(1)
            current_nombre = m.group(2).strip()
            data_taken     = False
            continue

        # Detectar fila de año (primera bajo el municipio = más reciente)
        if current_cod and not data_taken:
            fecha_m = re.search(r'1 de enero de (\d{4})', cell)
            if fecha_m and row[1] is not None:
                records.append({
                    'cod_ine':      current_cod,
                    'nombre_edad':  current_nombre,
                    'anio_edad':    int(fecha_m.group(1)),
                    # Total (españoles + extranjeros, ambos sexos)
                    'edad_todas':   _int(row[1]),
                    'pob_0_15':     _int(row[2]),
                    'pob_16_64':    _int(row[3]),   # ← EDAD LABORAL
                    'pob_65_mas':   _int(row[4]),
                    # Solo españoles
                    'esp_todas':    _int(row[5]),
                    'esp_0_15':     _int(row[6]),
                    'esp_16_64':    _int(row[7]),
                    'esp_65_mas':   _int(row[8]),
                    # Solo extranjeros
                    'ext_todas':    _int(row[9]),
                    'ext_0_15':     _int(row[10]),
                    'ext_16_64':    _int(row[11]),
                    'ext_65_mas':   _int(row[12]),
                })
                data_taken = True

    df = pd.DataFrame(records)
    if df.empty:
        print("  ⚠ No se encontraron datos de municipio en el fichero.")
        return df

    provs = df['cod_ine'].str[:2].unique()
    print(f"  → {len(df):,} municipios  |  provincias: {sorted(provs)}  |  año ref: {df['anio_edad'].iloc[0]}")
    return df


def _int(v):
    """Convierte a int o devuelve None si es nulo."""
    return int(v) if v is not None and not (isinstance(v, float) and pd.isna(v)) else None


df_edad = cargar_33861(DATA_DIR / '33861.xlsx')
df_edad.head(3)

In [ ]:
if not df_edad.empty:
    total = df_edad['pob_16_64'].sum()
    print(f"Municipios con datos de edad: {len(df_edad):,}")
    print(f"Año de referencia:            {df_edad['anio_edad'].iloc[0]}")
    print(f"Provincias cubiertas:         {sorted(df_edad['cod_ine'].str[:2].unique())}")
    print(f"Total pob. 16-64 en muestra:  {total:,.0f}")
    print()
    print("Primeras filas:")
    display(df_edad[['cod_ine','nombre_edad','anio_edad','pob_0_15','pob_16_64','pob_65_mas']].head(5))
else:
    print("El fichero 33861 no tiene datos. Verifica que el fichero es correcto.")

## 4. [Recomendado] Obtener superficie km² desde la API del INE

La superficie municipal **no está en los ficheros Excel** de pobmun ni en el 33861.  
Viene de la **tabla 2879** de la API JSON del INE, que devuelve superficie, densidad 
y población por municipio.

> **Nota:** Esta celda requiere conexión a internet y puede tardar 5–15 segundos  
> por provincia (la tabla tiene ~25.000 filas). Si prefieres trabajar offline,  
> salta a la sección 5 — la densidad quedará como `NaN` y podrás filtrarla manualmente.

### Cómo funciona la API del INE (tabla 2879)

```
GET https://servicios.ine.es/wstempus/js/ES/DATOS_TABLA/2879?nult=1
```

Devuelve un JSON con estructura:
```json
[
  {
    "Nombre": "02001 Abengibre. Superficie en km2",
    "Data": [{"Valor": 59.33, "NombrePeriodo": "2024"}]
  },
  ...
]
```

El código de municipio (5 dígitos) está embebido en el campo `Nombre`.

In [ ]:
import requests

def obtener_superficie_api(timeout: int = 30) -> pd.DataFrame:
    """
    Descarga superficie (km²) y densidad de todos los municipios desde
    la tabla 2879 del INE.

    Tabla 2879: 'Densidad de población por municipio'
    Operación: Cifras Oficiales de Población / Padrón Municipal
    Ref. datos: 1 enero 2024
    URL API: servicios.ine.es/wstempus/js/ES/DATOS_TABLA/2879?nult=1
    """
    url = 'https://servicios.ine.es/wstempus/js/ES/DATOS_TABLA/2879?nult=1'
    print(f"Consultando tabla 2879... ", end='')
    try:
        r = requests.get(url, timeout=timeout)
        r.raise_for_status()
        raw = r.json()
    except Exception as e:
        print(f"ERROR: {e}")
        print("Continuando sin datos de superficie.")
        return pd.DataFrame(columns=['cod_ine', 'sup_km2', 'densidad_2024', 'anio_sup'])

    print(f"{len(raw):,} registros")

    rows = []
    for item in raw:
        nombre = str(item.get('Nombre', ''))
        valor  = item.get('Data', [{}])[0].get('Valor')
        anio   = item.get('Data', [{}])[0].get('NombrePeriodo', '2024')
        m = re.search(r'\b(\d{5})\b', nombre)
        if not m or valor is None:
            continue
        cod = m.group(1)
        nl  = nombre.lower()
        rows.append({'cod_ine': cod, 'campo': nl, 'valor': valor, 'anio': anio})

    df_raw = pd.DataFrame(rows)
    if df_raw.empty:
        print("  ⚠ No se extrajeron datos.")
        return pd.DataFrame(columns=['cod_ine', 'sup_km2', 'densidad_2024', 'anio_sup'])

    # Pivotar: superficie y densidad en columnas separadas
    sup  = df_raw[df_raw['campo'].str.contains('superficie')].rename(columns={'valor':'sup_km2', 'anio':'anio_sup'})
    dens = df_raw[df_raw['campo'].str.contains('densidad')].rename(columns={'valor':'densidad_2024'})

    df_sup = sup[['cod_ine','sup_km2','anio_sup']].drop_duplicates('cod_ine')
    df_den = dens[['cod_ine','densidad_2024']].drop_duplicates('cod_ine')
    df_out = df_sup.merge(df_den, on='cod_ine', how='outer')

    print(f"  → {len(df_out):,} municipios con superficie  |  ref. {df_out['anio_sup'].iloc[0] if len(df_out) else 'N/A'}")
    return df_out


# Descomentar para ejecutar (requiere internet):
df_sup = obtener_superficie_api()
df_sup.head(3)

## 5. Cruce de fuentes — tabla maestra

Unimos las tres fuentes en un único DataFrame:

| Fuente | Clave join | Filas esperadas |
|--------|-----------|-----------------|
| `df_pob` (pobmun25) | `cod_ine` | 8.132 |
| `df_sup` (API 2879) | `cod_ine` | ~8.100 (algunos municipios sin dato) |
| `df_edad` (33861) | `cod_ine` | Solo la provincia descargada |

El merge es `left` para conservar todos los municipios de `df_pob`.

In [ ]:
def construir_tabla_maestra(df_pob, df_sup, df_edad):
    """
    Cruza las tres fuentes en un único DataFrame limpio.

    Parámetros
    ----------
    df_pob  : DataFrame de pobmun25 (8132 filas, todas las provincias)
    df_sup  : DataFrame de tabla 2879 (superficie + densidad)
    df_edad : DataFrame de tabla 33861 (grupos de edad, puede ser parcial)

    Devuelve
    --------
    DataFrame con las columnas descritas abajo.
    """
    # 1. Base: población oficial 2025
    df = df_pob.copy()

    # 2. Superficie y densidad (API 2879, ref. 2024)
    if not df_sup.empty:
        df = df.merge(df_sup[['cod_ine','sup_km2','densidad_2024','anio_sup']],
                      on='cod_ine', how='left')
        # Recalcular densidad con pob. 2025 (más actual que la densidad de la tabla)
        df['densidad'] = (df['pob_total'] / df['sup_km2']).round(2)
        df['fuente_sup'] = 'INE tabla 2879 (ref. ' + df['anio_sup'].fillna('2024') + ')'
    else:
        df['sup_km2']      = None
        df['densidad_2024'] = None
        df['densidad']     = None
        df['fuente_sup']   = 'no disponible (sin conexión)'

    # 3. Grupos de edad (tabla 33861, ref. 2022 o el año del fichero)
    if not df_edad.empty:
        cols_edad = ['cod_ine','anio_edad','pob_0_15','pob_16_64','pob_65_mas',
                     'esp_16_64','ext_16_64']
        df = df.merge(df_edad[cols_edad], on='cod_ine', how='left')
        df['fuente_edad'] = df['anio_edad'].apply(
            lambda a: f'INE tabla 33861 (ref. {int(a)})' if pd.notna(a) else 'no disponible'
        )
    else:
        for c in ['anio_edad','pob_0_15','pob_16_64','pob_65_mas','esp_16_64','ext_16_64']:
            df[c] = None
        df['fuente_edad'] = 'no disponible'

    # 4. Porcentajes de edad (solo donde hay datos)
    df['pct_0_15']   = (df['pob_0_15']   / df['pob_total'] * 100).round(1)
    df['pct_16_64']  = (df['pob_16_64']  / df['pob_total'] * 100).round(1)
    df['pct_65_mas'] = (df['pob_65_mas'] / df['pob_total'] * 100).round(1)

    # 5. Clasificación Ley 45/2007
    #    Ambos criterios deben cumplirse simultáneamente
    tiene_sup  = df['densidad'].notna()
    crit_pob   = df['pob_total'] < 30_000
    crit_dens  = df['densidad']  < 100
    df['cumple_pob']   = crit_pob
    df['cumple_dens']  = tiene_sup & crit_dens
    df['es_zona_rural'] = crit_pob & tiene_sup & crit_dens
    df['sin_superficie'] = ~tiene_sup  # municipios que no se pueden clasificar

    # 6. Ordenar columnas de forma lógica
    col_order = [
        'cod_ine', 'nombre', 'cpro', 'provincia', 'ccaa',
        'pob_total', 'hombres', 'mujeres', 'anio_pob',
        'sup_km2', 'densidad', 'densidad_2024', 'fuente_sup',
        'cumple_pob', 'cumple_dens', 'es_zona_rural', 'sin_superficie',
        'anio_edad', 'pob_0_15', 'pct_0_15',
        'pob_16_64', 'pct_16_64',
        'pob_65_mas', 'pct_65_mas',
        'esp_16_64', 'ext_16_64',
        'fuente_edad',
    ]
    df = df[[c for c in col_order if c in df.columns]]
    return df


df = construir_tabla_maestra(df_pob, df_sup, df_edad)
print(f"Tabla maestra: {len(df):,} filas × {len(df.columns)} columnas")
df.head(3)

In [ ]:
# Resumen de cobertura
print("=" * 55)
print("COBERTURA DE LA TABLA MAESTRA")
print("=" * 55)
print(f"Municipios totales:              {len(df):>7,}")
print(f"  Con superficie km²:            {df['sup_km2'].notna().sum():>7,}")
print(f"  Sin superficie (no clasif.):   {df['sin_superficie'].sum():>7,}")
print()
print(f"  Cumplen criterio población:    {df['cumple_pob'].sum():>7,}  (<30.000 hab)")
print(f"  Cumplen criterio densidad:     {df['cumple_dens'].sum():>7,}  (<100 hab/km²)")
print(f"  → ZONA RURAL (ambos):          {df['es_zona_rural'].sum():>7,}")
print()
print(f"  Con datos de grupos de edad:   {df['pob_16_64'].notna().sum():>7,}")
print()
print("Referencias temporales:")
print(f"  Población:        {df['anio_pob'].iloc[0]}")
sup_ref = df['fuente_sup'].iloc[0] if len(df) else 'N/A'
print(f"  Superficie:       {sup_ref}")
edad_muestras = df[df['anio_edad'].notna()]['anio_edad']
if len(edad_muestras):
    print(f"  Edad (grupos):    {int(edad_muestras.iloc[0])}")
else:
    print(f"  Edad (grupos):    no disponible")

## 6. Filtros — tabla de zonas rurales

Aquí aplicas los criterios que necesites. Puedes modificar los parámetros
o añadir filtros adicionales (CCAA, provincia, umbral de envejecimiento, etc.)

In [ ]:
# ── Parámetros del filtro ──────────────────────────────────────────────────────
# Modifica estos valores según tus necesidades

POB_MAX   = 30_000   # criterio Ley 45/2007
DENS_MAX  = 100      # criterio Ley 45/2007 (hab/km²)

# Filtros adicionales opcionales (descomenta los que necesites):
# CCAA_FILTRO   = ['Castilla y León', 'Extremadura']  # solo estas CCAA
# PROV_FILTRO   = ['42', '49']                        # solo estas provincias (cod INE)
# POB_MIN       = 0                                   # excluir municipios deshabitados
# PCT_65_MIN    = 30                                  # municipios muy envejecidos (>30% mayores)

# ── Aplicar filtro zona rural ──────────────────────────────────────────────────
df_rural = df[df['es_zona_rural']].copy()

# Descomenta para aplicar filtros adicionales:
# if 'CCAA_FILTRO' in dir():
#     df_rural = df_rural[df_rural['ccaa'].isin(CCAA_FILTRO)]
# if 'PROV_FILTRO' in dir():
#     df_rural = df_rural[df_rural['cpro'].isin(PROV_FILTRO)]

print(f"Zonas rurales (Ley 45/2007): {len(df_rural):,} municipios")
print(f"  de un total de:            {len(df):,} municipios con superficie conocida")
print(f"  Porcentaje:                {len(df_rural)/df['sup_km2'].notna().sum()*100:.1f}%")
print()
print("Distribución por CCAA:")
res = (df_rural.groupby('ccaa')
       .agg(n_municipios=('cod_ine','count'),
            pob_rural=('pob_total','sum'),
            densidad_media=('densidad','mean'))
       .sort_values('n_municipios', ascending=False)
       .round(1))
display(res)

In [ ]:
# Vista de los municipios rurales con datos de edad
cols_vista = ['cod_ine','nombre','provincia','ccaa',
              'pob_total','sup_km2','densidad',
              'pob_16_64','pct_16_64','pob_65_mas','pct_65_mas',
              'anio_edad','fuente_sup','fuente_edad']

df_rural_vista = df_rural[[c for c in cols_vista if c in df_rural.columns]]

# Estadísticas de edad solo en municipios con datos
con_edad = df_rural_vista[df_rural_vista['pob_16_64'].notna()]
print(f"Municipios rurales con datos de edad: {len(con_edad):,}")
if len(con_edad):
    print(f"  Pob. 16-64 total:  {con_edad['pob_16_64'].sum():,.0f}")
    print(f"  % medio 16-64:     {con_edad['pct_16_64'].mean():.1f}%")
    print(f"  % medio 65+:       {con_edad['pct_65_mas'].mean():.1f}%")
print()
print("Primeras filas:")
display(df_rural_vista.head(10))

## 7. Análisis descriptivo

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Histograma densidad (zona rural)
ax = axes[0]
df_rural['densidad'].dropna().plot.hist(bins=40, ax=ax, color='#2d7a4f', alpha=.8)
ax.axvline(100, color='#8b1a0a', ls='--', lw=1.2, label='Límite 100 hab/km²')
ax.set_xlabel('Densidad (hab/km²)')
ax.set_title('Densidad — municipios rurales')
ax.legend(fontsize=9)

# Distribución por CCAA
ax2 = axes[1]
top = df_rural.groupby('ccaa')['cod_ine'].count().sort_values(ascending=True).tail(12)
top.plot.barh(ax=ax2, color='#2d7a4f', alpha=.85)
ax2.set_xlabel('Nº municipios rurales')
ax2.set_title('Municipios rurales por CCAA')
ax2.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))

# % pob. rural por tramos de tamaño
ax3 = axes[2]
bins = [0, 500, 1000, 5000, 10000, 30000]
labels = ['<500', '500-1K', '1K-5K', '5K-10K', '10K-30K']
df_rural['tramo'] = pd.cut(df_rural['pob_total'], bins=bins, labels=labels)
tramo_count = df_rural['tramo'].value_counts().sort_index()
tramo_count.plot.bar(ax=ax3, color='#1a4e8c', alpha=.85)
ax3.set_xlabel('Habitantes')
ax3.set_title('Municipios rurales por tamaño de población')
ax3.tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.savefig('zonas_rurales_analisis.png', dpi=150, bbox_inches='tight')
plt.show()
print("Gráfico guardado: zonas_rurales_analisis.png")

In [ ]:
# Estadísticas descriptivas de los municipios rurales
print("ESTADÍSTICAS — MUNICIPIOS ZONA RURAL")
print("=" * 50)

stats_cols = ['pob_total', 'sup_km2', 'densidad']
print(df_rural[stats_cols].describe().round(1).to_string())

if df_rural['pob_16_64'].notna().any():
    print()
    print("GRUPOS DE EDAD (municipios con dato disponible)")
    print("=" * 50)
    edad_stats = ['pob_0_15','pob_16_64','pob_65_mas','pct_16_64','pct_65_mas']
    print(df_rural[[c for c in edad_stats if c in df_rural.columns]].describe().round(1).to_string())

## 8. Exportar resultados

In [ ]:
# ── Tabla completa (todos los municipios con clasificación) ──────────────────
out_completa = 'municipios_espana_clasificados.csv'
df.to_csv(out_completa, index=False, sep=';', encoding='utf-8-sig')
print(f"✓ {out_completa}  ({len(df):,} filas)")

# ── Solo municipios rurales ────────────────────────────────────────────────────
out_rural = 'municipios_zona_rural.csv'
df_rural.to_csv(out_rural, index=False, sep=';', encoding='utf-8-sig')
print(f"✓ {out_rural}  ({len(df_rural):,} filas)")

# ── Tabla rural con todos los campos de edad ────────────────────────────────
out_edad = 'municipios_zona_rural_edad.csv'
df_con_edad = df_rural[df_rural['pob_16_64'].notna()].copy()
df_con_edad.to_csv(out_edad, index=False, sep=';', encoding='utf-8-sig')
print(f"✓ {out_edad}  ({len(df_con_edad):,} filas — solo con datos de edad)")

print()
print("Separador: punto y coma (;) · Codificación: UTF-8 con BOM (compatible Excel)")
print()
print("Columnas exportadas:")
for col in df.columns:
    print(f"  {col}")

## 9. Guía de actualización de datos

### Cuándo actualizar cada fuente

| Fuente | Cuándo | Cómo |
|--------|--------|------|
| `pobmun.zip` | Cada diciembre (publica pob. 1 enero) | Descarga y reemplaza en `data/` |
| Tabla 2879 (API) | Automático al ejecutar celda 4 | La API devuelve siempre el último dato |
| `33861.xlsx` | Cuando el INE publique datos de 2023+ | Descarga desde INEbase tabla 33861 |

### Para añadir datos de edad de más provincias

El fichero `33861.xlsx` que tienes solo tiene **Navarra**.  
Para añadir más provincias hay dos métodos:

**Método A — Descarga por provincia (tedioso pero fiable):**
```python
# Para cada provincia, descarga desde INEbase > Padrón Continuo > Tabla 33861
# Selecciona: Provincia = XX, Sexo = Ambos sexos, Nacionalidad = Total
# Descarga como Excel, guarda en data/33861_XX.xlsx
# Luego:
dfs = [cargar_33861(p) for p in DATA_DIR.glob('33861_*.xlsx')]
df_edad_nacional = pd.concat(dfs, ignore_index=True)
```

**Método B — Descarga nacional completa (una sola vez):**
1. Ve a: https://www.ine.es/jaxiT3/Tabla.htm?t=33861
2. Selecciona: Sexo = Ambos sexos · Nacionalidad = Total · Municipios = Todos · Año = último
3. Descarga como Excel (puede tardar, son ~8000 municipios × 20 años)
4. Guarda como `data/33861_nacional.xlsx`
5. Ejecuta `df_edad = cargar_33861(DATA_DIR / '33861_nacional.xlsx')`

### Cita recomendada
```
Instituto Nacional de Estadística (INE). Cifras Oficiales de Población de los
municipios españoles: Revisión del Padrón Municipal a 1 de enero de 2025.
Publicado 11 de diciembre de 2025. Disponible en: www.ine.es
Licencia: CC BY 4.0
```